In [3]:
from pystac_client import Client
import planetary_computer
import stackstac
import numpy as np
from tqdm import tqdm
from dask.diagnostics import ProgressBar
import os

# Reduce GDAL stress (very important for Azure COG)
os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"
os.environ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS"] = ".tif"
os.environ["GDAL_NUM_THREADS"] = "1"

# ----------------------------------
# 1️⃣ Search Sentinel
# ----------------------------------
catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")

bbox = [77.90, 30.20, 78.20, 30.45]

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2023-11-01/2023-12-31",
    query={"eo:cloud_cover": {"lt": 10}},
)

items = list(search.get_items())
print("Scenes found:", len(items))

if len(items) == 0:
    raise ValueError("No scenes found")

# Sign
signed_items = [planetary_computer.sign(item) for item in items]

# ----------------------------------
# 2️⃣ Process Scene-by-Scene
# ----------------------------------
all_scenes = []

for item in tqdm(signed_items, desc="Processing scenes"):

    print(f"\nProcessing scene: {item.id}")

    # Extract zone from MGRS tile
    tile = item.id.split("_")[5]      # T44RKU
    zone = int(tile[1:3])             # 44
    epsg = 32600 + zone               # Northern hemisphere

    print("Using EPSG:", epsg)

    stack = stackstac.stack(
        [item],
        assets=["B02", "B03", "B04", "B08"],
        resolution=10,
        bounds_latlon=bbox,
        epsg=epsg,            # 🔥 REQUIRED
        chunksize=512,
    )

    stack = stack.squeeze("time")

    with ProgressBar():
        stack = stack.compute()

    sentinel = stack.values.astype(np.float32) / 10000.0

    print("Scene shape:", sentinel.shape)

    print("Min/Max:", sentinel.min(), sentinel.max())

    all_scenes.append({
        "id": item.id,
        "date": item.properties["datetime"],
        "data": sentinel
    })

print("\nFinished processing all scenes.")

Scenes found: 24


Processing scenes:   0%|          | 0/24 [00:00<?, ?it/s]


Processing scene: S2A_MSIL2A_20231225T053231_R105_T44RKU_20231225T100619
Using EPSG: 32602
[########################################] | 100% Completed | 13m 3ss


Processing scenes:   4%|▍         | 1/24 [13:04<5:00:51, 784.83s/it]

Scene shape: (4, 6701, 6663)
Min/Max: nan nan

Processing scene: S2A_MSIL2A_20231225T053231_R105_T43RGP_20231225T100620
Using EPSG: 32602
[###############                         ] | 39% Completed | 538.72 s


Processing scenes:   4%|▍         | 1/24 [1:32:20<35:23:42, 5540.11s/it]


RuntimeError: Error reading Window(col_off=4096, row_off=6656, width=512, height=45) from 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/43/R/GP/2023/12/25/S2A_MSIL2A_20231225T053231_N0510_R105_T43RGP_20231225T100620.SAFE/GRANULE/L2A_T43RGP_A044432_20231225T053231/IMG_DATA/R10m/T43RGP_20231225T053231_B08_10m.tif?st=2026-02-18T18%3A19%3A42Z&se=2026-02-19T19%3A04%3A42Z&sp=rl&sv=2025-07-05&sr=c&skoid=9c8ff44a-6a2c-4dfb-b298-1c9212f64d9a&sktid=72f988bf-86f1-41af-91ab-2d7cd011db47&skt=2026-02-19T04%3A28%3A58Z&ske=2026-02-26T04%3A28%3A58Z&sks=b&skv=2025-07-05&sig=GoXFt/kU0wEF3Jsae1DvXG25r6w67AWUwBm4h3HXKlE%3D': RasterioIOError('Read failed. See previous exception for details.')